# TOPSIS法 — 逼近理想解排序法

**原理**：通过计算各方案与正理想解（最优）和负理想解（最劣）的距离，按相对贴近度排序。

**步骤**：正向化 → 向量归一化 → 加权 → 计算正/负理想解 → 贴近度 → 排序


In [ ]:
import numpy as np
import sys; sys.path.insert(0, '..')
from src.utils.matrix import positive_transform, vector_normalize
from src.algorithms.topsis import ideal_solutions, calculate_closeness
from src.utils.plot import Plotter


## 示例：供应商评选

3 家供应商 × 4 项指标：

| 指标 | 类型 |
|------|------|
| 价格 | 极小型（越低越好） |
| 质量 | 极大型（越高越好） |
| 交期 | 极小型（越短越好） |
| 服务 | 极大型（越高越好） |


In [ ]:
# 原始评价矩阵：3 方案 × 4 指标
matrix = np.array([
    [80.0, 90.0, 3.0, 85.0],   # 供应商 A
    [85.0, 85.0, 5.0, 90.0],   # 供应商 B
    [90.0, 80.0, 2.0, 80.0],   # 供应商 C
])

kinds = [2, 1, 2, 1]  # 价格/交期=极小型，质量/服务=极大型
labels = ["供应商A", "供应商B", "供应商C"]
ind_labels = ["价格", "质量", "交期", "服务"]

print("原始评价矩阵：")
print(matrix)


In [ ]:
# 步骤 1：正向化 — 所有指标统一为极大型
converted = positive_transform(matrix, kinds)
print("正向化后的矩阵（数值越大越优）：")
print(np.round(converted, 4))


In [ ]:
# 步骤 2：向量归一化 — 每列除以欧几里得范数
normalized = vector_normalize(converted)
print("归一化矩阵（每列范数为 1）：")
print(np.round(normalized, 4))


In [ ]:
# 步骤 3：加权规范化矩阵（等权重）
weights = np.ones(4) / 4
weighted = normalized * weights
print("加权规范化矩阵：")
print(np.round(weighted, 4))


In [ ]:
# 步骤 4：确定正理想解与负理想解
v_pos, v_neg = ideal_solutions(weighted)
print(f"正理想解 V+（各指标最优值）: {np.round(v_pos, 4)}")
print(f"负理想解 V-（各指标最劣值）: {np.round(v_neg, 4)}")


In [ ]:
# 步骤 5：计算贴近度并排序
closeness = calculate_closeness(weighted, v_pos, v_neg)
ranks = np.argsort(-closeness) + 1  # 贴近度越大越优

print("=== 评价结果 ===")
for name, c, r in zip(labels, closeness, ranks):
    print(f"  {name}：贴近度 = {c:.4f}，排名 = {r}")

best = np.argmax(closeness)
print(f"\n✓ 最优方案：{labels[best]}")


In [ ]:
# 可视化
fig, axes = Plotter.subplots(1, 3, figsize=(14, 4))

Plotter.heatmap(converted, row_labels=labels, col_labels=ind_labels,
                ax=axes[0], title="正向化矩阵", fmt=".1f")
Plotter.heatmap(weighted, row_labels=labels, col_labels=ind_labels,
                ax=axes[1], title="加权规范矩阵", fmt=".4f")
Plotter.bar(labels, closeness, ax=axes[2], title="相对贴近度")

Plotter.save("../data/topsis_result.png")
print("图片已保存至 data/topsis_result.png")
